# Quickstart Guide

T2Fpharm provides two main routes for generating pharmacophores:
1. **Complex-based pharmacophores**
   can be generated from the structure of target–ligand complexes.
   This uses the PLIP library under the hood.
2. **Target-based pharmacophores**
   can be generated from 3D scalar fields using different perception methods.

T2FPharm has a highly modular design,
allowing users to generate pharmacophores
from various input data.
Below is a minimal example starting only from the PDB ID of a protein-ligand complex
to generate both a complex-based and a target-based pharmacophore.
For other possible scenarios and more details see the other provided notebooks.

All functionalities are accessible from the top-level `t2fpharm` package,
which is the only module that needs to be imported:

In [1]:
import sciapi  # to fetch PDB files from the RCSB webserver
import t2fpharm

For this route, we require the following inputs:

In [2]:
# PDB ID of the complex
PDB_ID: str = "1AQ1"
# Residue name, chain ID, and residue sequence number of the ligand of interest
LIGAND_RES_NAME: str = "STU"
LIGAND_CHAIN_ID: str = "A"
LIGAND_RES_SEQ: int = 299

## System Generation

The first step is to create a `System` object containing structural data of the target–ligand complex.
First, obtain a PDB file:

In [3]:
pdb_raw: str = sciapi.pdb.file.entry(pdb_id=PDB_ID, file_format="pdb").decode()

Fix the PDB file (add hydrogens and other missing atoms):

In [4]:
pdb_fixed: str = t2fpharm.system.fix_pdb(file=pdb_raw)[0]

Create a chemical system from the fixed complex structure:

In [5]:
rcomplex: t2fpharm.system.System = t2fpharm.system.from_pdb(pdb_fixed)

For more information, see the notebook [`2_system.ipynb`](./2_system.ipynb).

## Pocket Generation

Next, we need to define the binding pocket.
Here, we use the co-crystalized ligand to define the pocket.
For this, we must define the following parameters:

In [6]:
# Desired grid spacing (in Å)
GRID_SPACING: float = 0.3
# Expansion radius (in Å) for each ligand atom
RADII_OFFSET: float = 2.7
# Radius (in Å) of the morphological opening operation to remove small artifacts
OPENING_RADIUS: float = 1

We can now generate a `t2fpharm.pocket.Pocket` object:

In [7]:
atoms = rcomplex.composition.atoms  # Atom data in the complex as a pandas.DataFrame object
ligand_mask = (  # Boolean mask to select the ligand atoms
    (atoms["res_name"] == LIGAND_RES_NAME) &
    (atoms["chain_id"] == LIGAND_CHAIN_ID) &
    (atoms["res_seq"] == LIGAND_RES_SEQ)
)
pocket: t2fpharm.pocket.Pocket = t2fpharm.pocket.from_ligand(
    system=rcomplex,
    ligand_mask=ligand_mask,
    ligand_radii_offset=RADII_OFFSET,  
    opening_radius=OPENING_RADIUS,  
    grid=GRID_SPACING,  
)

For more information, see the notebook [`3_pocket.ipynb`](./3_pocket.ipynb).

## Complex-Based Pharmacophore Generation

With the complex structure and binding pocket at hand,
we can create a complex-based pharmacophore using the `t2fpharm.pharm.from_complex()` function.
For this, a feature type ID can be provided for each PLIP interaction type (if not, default IDs are used).
To exclude a certain interaction type, the corresponding feature type can be set to `None`.
For example:

In [18]:
pharm_complex: t2fpharm.pharm.Pharmacophore = t2fpharm.pharm.from_complex(
    pdb_fixed,
    pocket=pocket,
    receptor=rcomplex,
    type_aromatic=None,
)

More information can be found in the function's docstring:

In [19]:
help(t2fpharm.pharm.from_complex)

Help on function from_complex in module t2fpharm.pharm:

from_complex(pdb_files: Union[str, bytes, pathlib.Path, Sequence], ligands: Optional[Sequence[tuple[str, int | str, int]]] = None, type_hbond_acceptor: str | None = 'OA', type_hbond_donor: str | None = 'HD', type_water_bridge_ligand_acceptor: str | None = 'OA', type_water_bridge_ligand_donor: str | None = 'HD', type_water_bridge_water_acceptor: str | None = 'OA', type_anion: str | None = 'e-', type_cation: str | None = 'e+', type_hydrophobic: str | None = 'C', type_aromatic: str | None = 'A', pocket: caddpy.pocket.pocket.Pocket | None = None, receptor: caddpy.chemsys.ChemicalSystem | None = None)
    Create a pharmacophore from a receptor–ligand complex.

    This function uses the PLIP library to analyze the interactions
    between the receptor and ligand(s) in the provided PDB files.
    The non-receptor interaction centers
    are then converted into pharmacophore features.

    Note that if any of the `type_*` parameters are

## Target-Based Pharmacophore Generation

### Receptor Isolation

To generate a target-based pharmacophore, we first need to isolate the receptor from the complex:

In [11]:
receptor: t2fpharm.system.System = rcomplex.select(rcomplex.composition.atoms["res_poly"])

### Field Generation

To generate energy fields using AutoGrid,
we first create a PDBQT file from the receptor:

In [12]:
receptor_pdbqt: str = receptor.to_pdbqt()

Fields can then be generated from the PDBQT file,
by defining a `Grid` (taken from the generated pocket)
and the AutoGrid ligand types to generate fields for:

In [13]:
field = t2fpharm.field.from_autogrid(
    grid=pocket.grid,
    receptor_files=receptor_pdbqt,
    ligand_types=("HD", "A", "C", "OA", "e-", "e+")
)

### Modeler Initialization

To generate target-based pharmacophores, we first create a modeler from the field and pocket:

In [14]:
modeler = t2fpharm.modeler(
    field=field,
    pocket=pocket,
    system=rcomplex  # for visualization; not required for modeling
)

### Pharmacophore Perception

The modeler provides two main methods to percieve pharmacophores using different algorithms.
For more information, see the notebook [`6_modeler.ipynb`](./6_modeler.ipynb).

#### Largest Peaks

The `largest_peaks()` method percieves pharmacophore features as largest extrema in the fields.
Note that here we are using a minimal set of input parameters that
may not produce the best results:

In [15]:
pharm_lp = modeler.largest_peaks(
    min_distance=2,
    max_features=10,
    threshold_value=0
)

#### CNN

The `cnn()` method percieves pharmacophore features by clustering field values using the CNN clustering algorithm.
Again, here we are using a minimal set of input parameters that may not produce the best results:

In [16]:
pharm_cnn = modeler.cnn(
    max_distance=2,
    min_neighbors=10,
    min_members=1,
    threshold_percentile=20,
)

## Pharmacophore Matching

Any two generated pharmacophores can be matched against each other. For example:

In [17]:
pharm_cnn.match(pharm_complex, max_distance=2)

,instance,type,label,target_instance,target_label,radius_sum,distance,match
0,0,OA,1,0,1,1.835600,12.144744,False
1,0,HD,1,0,1,8.085104,6.027122,False
2,0,HD,2,0,2,1.146083,7.368232,False
3,0,C,1,0,<NA>,NaN,NaN,False
4,0,C,2,0,<NA>,NaN,NaN,False
5,0,C,3,0,<NA>,NaN,NaN,False
6,0,C,4,0,<NA>,NaN,NaN,False
7,0,C,5,0,<NA>,NaN,NaN,False
8,0,C,6,0,<NA>,NaN,NaN,False
9,0,C,7,0,<NA>,NaN,NaN,False


For more information about `Pharmacophore` objects' methods and attributes,
see the notebook [`7_pharmacophore.ipynb`](./7_pharmacophore.ipynb).